# 05 — Retrieval-Augmented Generation (RAG) Pipeline

Objective:
Enhance biomedical QA by retrieving relevant documents
and augmenting input before classification.

## Step 1 - SetUp 

In [1]:
import torch
import numpy as np
import faiss

from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

## Step 2 - Load dataset again

In [2]:
from datasets import load_from_disk

dataset = load_from_disk("data/processed/pubmedqa")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision', 'input_text', 'label'],
        num_rows: 800
    })
    test: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision', 'input_text', 'label'],
        num_rows: 200
    })
})


## Step 3 — Prepare documents

In [3]:
documents = []

for item in dataset["train"]:
    context = item["context"]
    long_answer = item["long_answer"]

    # 🔹 Fix context format
    if isinstance(context, dict):
        context_text = " ".join(context.get("contexts", []))
    else:
        context_text = context

    # 🔹 Add ONLY clean versions
    documents.append(context_text)
    documents.append(long_answer)

## Step 4 — Build embedder

In [4]:
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 5 — Build FAISS index

In [5]:
doc_embeddings = embedder.encode(documents, show_progress_bar=True)

index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings))

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

## Step 6 - Retrieval function

In [6]:
def retrieve_docs(query, k=3):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(query_embedding, k)
    retrieved = [documents[i] for i in indices[0]]
    return retrieved

## Step 7 - Load model + tokenizer

In [7]:
model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    "../models/pubmedbert_model"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Step 8 - QA function

In [8]:
def answer_question(question):

    docs = retrieve_docs(question)

    # Join docs with separator so model can distinguish between them
    evidence_text = " [SEP] ".join(docs)

    # Combine question with evidence
    combined_text = question + " [SEP] " + evidence_text

    inputs = tokenizer(
        combined_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    outputs = model(**inputs)
    logits = outputs.logits
    pred = torch.argmax(logits, dim=1).item()

    labels = ["yes", "no", "maybe"]

    return {
        "answer": labels[pred],
        "evidence": docs,
        "retrieved_count": len(docs)
    }

## Step 9 - Run Test 

In [9]:
result = answer_question("Does aspirin reduce heart attack risk?")

print("Answer:", result["answer"])
print("\nEvidence:")
for doc in result["evidence"]:
    print("-", doc[:200])

Answer: yes

Evidence:
- Admission to a hospital ranked high on the list of "America's Best Hospitals" was associated with lower 30-day mortality among elderly patients with acute myocardial infarction. A substantial portion 
- Recent studies have demonstrated that statins have pleiotropic effects, including anti-inflammatory effects and atrial fibrillation (AF) preventive effects. The objective of this study was to assess t
- Our study indicated that preoperative statin therapy seems to reduce AF development after CABG.


## Evaluation on the test set

In [12]:
from sklearn.metrics import classification_report, accuracy_score

predictions = []
true_labels = []

print("Running evaluation on test set...")

for item in dataset["test"]:
    result = answer_question(item["question"])
    predictions.append(result["answer"])
    true_labels.append(item["final_decision"])

print(f"\nTotal questions evaluated: {len(predictions)}")
print(f"\nAccuracy: {accuracy_score(true_labels, predictions):.4f}")
print("\nDetailed Report:")
print(classification_report(true_labels, predictions, target_names=["yes", "no", "maybe"]))

Running evaluation on test set...

Total questions evaluated: 200

Accuracy: 0.4700

Detailed Report:
              precision    recall  f1-score   support

         yes       0.33      0.04      0.07        25
          no       0.35      0.24      0.28        72
       maybe       0.51      0.74      0.61       103

    accuracy                           0.47       200
   macro avg       0.40      0.34      0.32       200
weighted avg       0.43      0.47      0.42       200

